# 🎨 Pipeline de Análise Colorimétrica (Pele e Pelo)
Este notebook executa o fluxo completo de:
1. **Detecção Dinâmica Híbrida** da mancha de amostras de pele (YOLO + Visão Computacional adaptativa).
2. **Recorte Sincronizado** de pares de imagem (Pele x Pelo).
3. **Extração de Características** de cor e textura em grid 4x4 (16 partes por amostra).
4. **Modelagem Estatística** e calibração para predição das coordenadas de cor ^*, a^*, b^*$.

In [1]:
# 1. Instalação e Importação de Bibliotecas
import os
import re
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error


print("Bibliotecas carregadas com sucesso!")

Bibliotecas carregadas com sucesso!


In [2]:
# 2. Configuração do Detector Híbrido de Manchas (YOLO + Visão Computacional)
CAMINHO_MODELO_YOLO = '/content/drive/MyDrive/Colorímetro/best.pt'
if not os.path.exists(CAMINHO_MODELO_YOLO):
    CAMINHO_MODELO_YOLO = '/media/workstation_2x_3080/HD 1TB/Famez_yolo/Colorímetro/best.pt'
if not os.path.exists(CAMINHO_MODELO_YOLO):
    CAMINHO_MODELO_YOLO = 'best.pt'

model_yolo = YOLO(CAMINHO_MODELO_YOLO) if os.path.exists(CAMINHO_MODELO_YOLO) else None

def detectar_mancha_hibrido(img_bgr, model_yolo=None):
    """
    Detecta a mancha de amostra de pele usando YOLO como primário (conf 0.15).
    Se o YOLO falhar (0 detecções), aplica segmentação adaptativa de visão computacional
    na abertura circular iluminada do tubo para garantir 100% de sucesso.
    """
    h_max, w_max = img_bgr.shape[:2]
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    
    # A. Detecção da Abertura do Tubo/Cone (Círculo iluminado)
    blur_luz = cv2.GaussianBlur(gray, (61, 61), 0)
    _, thresh_luz = cv2.threshold(blur_luz, 60, 255, cv2.THRESH_BINARY)
    cnts_luz, _ = cv2.findContours(thresh_luz, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    bbox_tubo = None
    if cnts_luz:
        c_luz = max(cnts_luz, key=cv2.contourArea)
        tx, ty, tw, th = cv2.boundingRect(c_luz)
        bbox_tubo = (tx, ty, tw, th)
        
    # B. Tentar YOLO (Predição de mancha)
    if model_yolo is not None:
        results = model_yolo(img_bgr, conf=0.15, verbose=False)
        if len(results[0].boxes) > 0:
            box = results[0].boxes[0].xyxy[0].cpu().numpy()
            x1, y1, x2, y2 = map(int, box)
            
            # Margem de segurança de 10%
            pad_w = int((x2 - x1) * 0.10)
            pad_h = int((y2 - y1) * 0.10)
            x1, y1 = max(0, x1 - pad_w), max(0, y1 - pad_h)
            x2, y2 = min(w_max, x2 + pad_w), min(h_max, y2 + pad_h)
            return (x1, y1, x2, y2), bbox_tubo, 'YOLO'

    # C. Fallback: Segmentação da Abertura do Colorímetro (100% garantia)
    if bbox_tubo:
        tx, ty, tw, th = bbox_tubo
        mx, my = int(tw * 0.10), int(th * 0.10)
        x1, y1 = tx + mx, ty + my
        x2, y2 = tx + tw - mx, ty + th - my
        return (x1, y1, x2, y2), bbox_tubo, 'CV_Abertura'

    # D. Fallback Final (Caso imagem atípica)
    return (0, 0, w_max, h_max), None, 'Imagem_Total'

print("Detector Híbrido inicializado com sucesso!")

NameError: name 'YOLO' is not defined

In [ ]:
# 3. Recorte Dinâmico e Sincronizado de Amostras (Pele e Pelo)
caminhos_busca = [
    '/content/drive/MyDrive/Colorímetro/Fotos colorímetro- fevereiro-2026',
    '/content/drive/MyDrive/Colorímetro/fotos colorímetro-dezembro-2025',
    '/content/drive/MyDrive/Colorímetro/Foto colorimetro-Julho2026',
    '/content/drive/MyDrive/Doutorado/Colorímetro/Fotos colorímetro- fevereiro-2026',
    '/content/drive/MyDrive/Doutorado/Colorímetro/fotos colorímetro-dezembro-2025',
    '/content/drive/MyDrive/Doutorado/Colorímetro/Foto colorimetro-Julho2026',
    '/media/workstation_2x_3080/HD 1TB/Famez_yolo/Colorímetro/Fotos colorímetro- fevereiro-2026',
    '/media/workstation_2x_3080/HD 1TB/Famez_yolo/Colorímetro/fotos colorímetro-dezembro-2025',
    '/media/workstation_2x_3080/HD 1TB/Famez_yolo/Colorímetro/Foto colorimetro-Julho2026'
]

pastas_existentes = [p for p in caminhos_busca if os.path.exists(p)]
if not pastas_existentes:
    print("Aviso: Nenhuma pasta de imagens encontrada nos caminhos padrão.")
    pastas_existentes = ['.']

path_yolo_sinc = 'Amostras_Sincronizadas_YOLO'
os.makedirs(path_yolo_sinc, exist_ok=True)

print(f"Processando {len(pastas_existentes)} pasta(s) de origem para: {path_yolo_sinc}")

sucessos_yolo = 0
falhas = 0

for pasta_origem in pastas_existentes:
    arquivos = os.listdir(pasta_origem)
    pele_map = {}
    pelo_map = {}
    for f in arquivos:
        if not f.lower().endswith(('.jpg', '.png', '.jpeg')): continue
        m = re.findall(r'\d+', f)
        if not m: continue
        aid = m[0]
        if 'pele' in f.lower(): pele_map[aid] = f
        elif 'pelo' in f.lower(): pelo_map[aid] = f
        
    ids_comuns = set(pele_map.keys()).intersection(set(pelo_map.keys()))
    
    for aid in ids_comuns:
        try:
            caminho_pele = os.path.join(pasta_origem, pele_map[aid])
            caminho_pelo = os.path.join(pasta_origem, pelo_map[aid])
            
            img_pele_bgr = cv2.imread(caminho_pele)
            img_pelo_bgr = cv2.imread(caminho_pelo)
            if img_pele_bgr is None or img_pelo_bgr is None: continue

            # Detectar mancha exata / abertura na PELE dinamicamente
            coords, bbox_tubo, metodo = detectar_mancha_hibrido(img_pele_bgr, model_yolo)
            x1, y1, x2, y2 = coords

            # Recortes sincronizados (PELE e PELO usam o mesmo recorte)
            pele_final = img_pele_bgr[y1:y2, x1:x2]
            pelo_final = img_pelo_bgr[y1:y2, x1:x2]

            # Salvar amostras recortadas
            cv2.imwrite(os.path.join(path_yolo_sinc, f'{aid}_PELE_yolo.jpg'), pele_final)
            cv2.imwrite(os.path.join(path_yolo_sinc, f'{aid}_PELO_yolo.jpg'), pelo_final)
            sucessos_yolo += 1
        except Exception as e:
            falhas += 1

print(f"Concluído: {sucessos_yolo} pares de amostras salvos em '{path_yolo_sinc}'.")

Processando 3 pasta(s) de origem para: Amostras_Sincronizadas_YOLO
Concluído: 132 pares de amostras salvos em 'Amostras_Sincronizadas_YOLO'.


In [ ]:
# 4. Carregamento dos Dados de Referência do Colorímetro (Ground Truth L*, a*, b*)
def carregar_dados_colorimetro():
    caminhos_planilhas = [
        '/content/drive/MyDrive/Colorímetro/Dados do colorímetro coleta fevereiro.csv',
        '/content/drive/MyDrive/Colorímetro/Dados do colorímetro coleta dezembro.csv',
        'Dados do colorímetro coleta fevereiro.csv',
        'Dados do colorímetro coleta dezembro.csv',
        '/media/workstation_2x_3080/HD 1TB/Famez_yolo/Colorímetro/Dados do colorímetro coleta fevereiro.csv',
        '/media/workstation_2x_3080/HD 1TB/Famez_yolo/Colorímetro/Dados do colorímetro coleta dezembro.csv'
    ]
    
    dfs = []
    for p in caminhos_planilhas:
        if not os.path.exists(p): continue
        if p.endswith('.csv'):
            linhas = []
            with open(p, encoding='utf-8', errors='ignore') as f:
                for line in f:
                    parts = line.strip().split(',')
                    if len(parts) >= 7 and parts[0].isdigit():
                        linhas.append(parts[:8])
            if len(linhas) > 0:
                df = pd.DataFrame(linhas)
                if df.shape[1] == 8:
                    df.columns = ['idx', 'Animal', 'L_pelo', 'a_pelo', 'b_pelo', 'L_pele', 'a_pele', 'b_pele']
                elif df.shape[1] == 7:
                    df.columns = ['Animal', 'L_pelo', 'a_pelo', 'b_pelo', 'L_pele', 'a_pele', 'b_pele']
                cols_num = ['L_pelo', 'a_pelo', 'b_pelo', 'L_pele', 'a_pele', 'b_pele']
                for c in cols_num:
                    df[c] = pd.to_numeric(df[c], errors='coerce')
                df['Animal'] = df['Animal'].astype(str)
                dfs.append(df)
                
    if dfs:
        df_merged = pd.concat(dfs, ignore_index=True).drop_duplicates(subset=['Animal'])
        return df_merged
    return pd.DataFrame()

df_colorimetro = carregar_dados_colorimetro()
print(f"Base ground-truth carregada: {len(df_colorimetro)} registros de animais.")
display(df_colorimetro.head(5))

Base ground-truth carregada: 47 registros de animais.


,idx,Animal,L_pelo,a_pelo,b_pelo,L_pele,a_pele,b_pele
0,1,2425,64.17,1.15,16.93,29.77,1.88,6.59
1,2,2415,74.42,-1.60,13.02,28.26,0.15,4.09
2,3,2383,69.36,-0.71,15.30,26.12,0.17,4.25
3,4,2372,68.10,-0.96,13.20,29.24,0.72,5.36
4,5,2427,68.04,-1.23,15.15,27.81,0.79,5.15


In [ ]:
# 5. Extração de Características de Cor e Textura (Grid 4x4 - 16 Partes por Amostra)
def extrair_features_textura(img_bgr, num_partes=9):
    if img_bgr is None or img_bgr.size == 0: return []

    crop_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    crop_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    crop_lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)

    h_c, w_c = crop_rgb.shape[:2]
    grid = int(np.sqrt(num_partes))
    step_h, step_w = h_c // grid, w_c // grid

    lista_features = []
    for i in range(grid):
        for j in range(grid):
            y1, y2 = i * step_h, (i + 1) * step_h if i < grid - 1 else h_c
            x1, x2 = j * step_w, (j + 1) * step_w if j < grid - 1 else w_c

            b_rgb = crop_rgb[y1:y2, x1:x2]
            b_hsv = crop_hsv[y1:y2, x1:x2]
            b_lab = crop_lab[y1:y2, x1:x2]
            if b_rgb.size == 0: continue

            rgb_m, rgb_s = np.median(b_rgb, axis=(0,1)), np.std(b_rgb, axis=(0,1))
            hsv_m, hsv_s = np.median(b_hsv, axis=(0,1)), np.std(b_hsv, axis=(0,1))
            lab_m, lab_s = np.median(b_lab, axis=(0,1)), np.std(b_lab, axis=(0,1))

            lista_features.append({
                'R': rgb_m[0], 'G': rgb_m[1], 'B': rgb_m[2],
                'H': hsv_m[0], 'S': hsv_m[1], 'V': hsv_m[2],
                'L_cv2': lab_m[0], 'a_cv2': lab_m[1], 'b_cv2': lab_m[2],
                'R_std': rgb_s[0], 'G_std': rgb_s[1], 'B_std': rgb_s[2],
                'H_std': hsv_s[0], 'S_std': hsv_s[1], 'V_std': hsv_s[2],
                'L_std': lab_s[0], 'a_std': lab_s[1], 'b_std': lab_s[2]
            })
    return lista_features

dataset_final = []
arquivos_sucesso = 0
path_amostras = 'Amostras_Sincronizadas_YOLO'
if not os.path.exists(path_amostras):
    path_amostras = '/media/workstation_2x_3080/HD 1TB/Famez_yolo/Colorímetro/Amostras_Sincronizadas_YOLO'

if os.path.exists(path_amostras):
    for arquivo in os.listdir(path_amostras):
        if not arquivo.lower().endswith(('.jpg', '.png')): continue
        m = re.findall(r'\d+', arquivo)
        if not m: continue
        animal_id = m[0]
        tipo = 'pele' if 'PELE' in arquivo else 'pelo'

        linha = df_colorimetro[df_colorimetro['Animal'] == animal_id]
        if linha.empty: continue

        if tipo == 'pele': l, a, b = linha[['L_pele', 'a_pele', 'b_pele']].iloc[0]
        else: l, a, b = linha[['L_pelo', 'a_pelo', 'b_pelo']].iloc[0]
        if pd.isna(l): continue

        img_bgr = cv2.imread(os.path.join(path_amostras, arquivo))
        if img_bgr is None: continue

        feats = extrair_features_textura(img_bgr, num_partes=16)
        for f in feats:
            f['Animal'] = animal_id
            f['L'] = l
            f['a'] = a
            f['b'] = b
            f['Tipo_Pele'] = 1 if tipo == 'pele' else 0
            dataset_final.append(f)
        arquivos_sucesso += 1

df_ml = pd.DataFrame(dataset_final)
print(f"Dataset ML montado: {arquivos_sucesso} fotos transformadas em {len(df_ml)} sub-amostras (Grid 3x3).")

Dataset ML montado: 94 fotos transformadas em 1504 sub-amostras (Grid 3x3).


In [ ]:
# 6. Treinamento e Validação da Inteligência Artificial (Métricas do Colorímetro)
if 'df_ml' in locals() and len(df_ml) > 10:
    colunas_x = [
        'R', 'G', 'B', 'H', 'S', 'V', 'L_cv2', 'a_cv2', 'b_cv2',
        'R_std', 'G_std', 'B_std', 'H_std', 'S_std', 'V_std',
        'L_std', 'a_std', 'b_std', 'Tipo_Pele'
    ]
    X = df_ml[colunas_x]
    y = df_ml[['L', 'a', 'b']]
    grupos = df_ml['Animal']

    # Validação cega por animal (impede vazamento de dados entre treino e teste)
    separador = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    idx_treino, idx_teste = next(separador.split(X, y, groups=grupos))

    X_train, X_test = X.iloc[idx_treino], X.iloc[idx_teste]
    y_train, y_test = y.iloc[idx_treino], y.iloc[idx_teste]

    print(f"Base de Treino: {X_train.shape[0]} amostras | Base de Validação Cega: {X_test.shape[0]} amostras.")

    modelo = ExtraTreesRegressor(n_estimators=300, max_depth=20, random_state=42)
    modelo.fit(X_train, y_train)

    previsoes = modelo.predict(X_test)
    erro_l = mean_absolute_error(y_test['L'], previsoes[:, 0])
    erro_a = mean_absolute_error(y_test['a'], previsoes[:, 1])
    erro_b = mean_absolute_error(y_test['b'], previsoes[:, 2])

    print("=======================================================")
    print("      RESULTADOS FINAIS DA ESTATÍSTICA DO COLORÍMETRO  ")
    print("=======================================================")
    print(f"Erro Médio Validado para L*: ±{erro_l:.2f}")
    print(f"Erro Médio Validado para a*: ±{erro_a:.2f}")
    print(f"Erro Médio Validado para b*: ±{erro_b:.2f}")
    print("=======================================================")
else:
    print("[Erro] Falta de dados para treinar o modelo.")

Base de Treino: 1184 amostras | Base de Validação Cega: 320 amostras.
      RESULTADOS FINAIS DA ESTATÍSTICA DO COLORÍMETRO  
Erro Médio Validado para L*: ±2.71
Erro Médio Validado para a*: ±0.95
Erro Médio Validado para b*: ±1.14
